# Working with clinical notes

Exercise notebook for the lesson **[Working with clinical notes](https://ramkumarmd.com/medical-ai/clinical-nlp/working-with-clinical-notes)** in the *Clinical NLP* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Clinical notes hold much of the information that never reaches structured fields, but they are hard for computers: they mix sections, use local abbreviations, record what is absent as often as what is present ('denies chest pain'), and repeat text copied forward from earlier notes. Before any model, split notes into sections, expand abbreviations carefully, detect negation and uncertainty, and watch for duplicated text. Simple rules handle a lot of this, and their failures show where better tools are needed.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Splitting a note into sections

A synthetic diabetes clinic note, split on its capitalised headings.

In [ ]:
import re

note = """HISTORY: 58M with T2DM x 10 yrs, HTN. c/o tingling in both feet x 3 mo. Denies chest pain. No SOB.
MEDICATIONS: Metformin 1 g BD. Amlodipine 5 mg OD.
EXAMINATION: Reduced monofilament sensation both feet. No ulcers. Pedal pulses present.
ASSESSMENT: Probable diabetic peripheral neuropathy. HbA1c 8.4%.
PLAN: Refer for DPN testing. Foot care advice. Review in 3 mo."""

def split_sections(text):
    """Split a note on headings written in capitals followed by a colon."""
    parts = re.split(r"^([A-Z][A-Z ]+):\s*", text, flags=re.MULTILINE)[1:]
    return {parts[i].title(): parts[i + 1].strip() for i in range(0, len(parts), 2)}

for name, body in split_sections(note).items():
    print(f"[{name}] {body}")

# [History] 58M with T2DM x 10 yrs, HTN. c/o tingling in both feet x 3 mo. Denies chest pain. No SOB.
# [Medications] Metformin 1 g BD. Amlodipine 5 mg OD.
# [Examination] Reduced monofilament sensation both feet. No ulcers. Pedal pulses present.
# [Assessment] Probable diabetic peripheral neuropathy. HbA1c 8.4%.
# [Plan] Refer for DPN testing. Foot care advice. Review in 3 mo.

### Expanding abbreviations

Continuing from the previous block: expand a small, curated abbreviation list.

In [ ]:
ABBREVIATIONS = {
    "T2DM": "type 2 diabetes mellitus", "HTN": "hypertension", "c/o": "complains of",
    "SOB": "shortness of breath", "BD": "twice daily", "OD": "once daily",
    "mo": "months", "yrs": "years", "DPN": "diabetic peripheral neuropathy",
}
def expand(text):
    for short, long in ABBREVIATIONS.items():
        text = re.sub(rf"(?<![\w/]){re.escape(short)}(?![\w/])", long, text)
    return text

print(expand(split_sections(note)["History"]))

# 58M with type 2 diabetes mellitus x 10 years, hypertension. complains of tingling in both feet x 3 months. Denies chest pain. No shortness of breath.

### Detecting negation

Continuing from the previous blocks: a NegEx-style rule over the expanded note.

In [ ]:
NEGATION_CUES = r"\b(no|denies|denied|without|negative for|absent)\b"
FINDINGS = ["chest pain", "shortness of breath", "ulcers", "tingling", "pedal pulses",
            "reduced monofilament sensation"]

def findings_with_status(text):
    """NegEx-style: a finding is negated if a negation cue appears earlier in the same sentence."""
    results = []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        s = sentence.lower()
        for f in FINDINGS:
            pos = s.find(f)
            if pos >= 0:
                negated = re.search(NEGATION_CUES, s[:pos]) is not None
                results.append((f, "absent" if negated else "present"))
    return results

for finding, status in findings_with_status(expand(note)):
    print(f"{status:<8} {finding}")

# present  tingling
# absent   chest pain
# absent   shortness of breath
# present  reduced monofilament sensation
# absent   ulcers
# present  pedal pulses

### Copy-forward text

How much of each synthetic note is identical to the previous one?

In [ ]:
import difflib

visit_1 = "Tingling in both feet for 3 months. No ulcers. Metformin 1 g twice daily. Plan: refer for neuropathy testing."
visit_2 = visit_1 + " Neuropathy test done: reduced sural nerve response."
visit_3 = visit_2                                           # copied forward unchanged

for name, a, b in [("visit 2 vs 1", visit_1, visit_2), ("visit 3 vs 2", visit_2, visit_3)]:
    print(f"{name}: {difflib.SequenceMatcher(None, a, b).ratio():.0%} identical")

# visit 2 vs 1: 81% identical
# visit 3 vs 2: 100% identical

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Twenty SYNTHETIC clinic notes: five made-up patients, four visits each, in date order. No real patients.
clinic_notes = [
    ("P1", 1, """HISTORY: 61F with T2DM x 8 yrs, HTN. c/o tingling in feet x 2 mo. Denies chest pain.
MEDICATIONS: Metformin 1 g BD.
EXAMINATION: Reduced monofilament sensation both feet. No ulcers.
ASSESSMENT: Possible diabetic peripheral neuropathy. Family history of kidney disease (mother).
PLAN: Refer for DPN testing. Review in 3 mo."""),
    ("P1", 2, """HISTORY: 61F with T2DM x 8 yrs, HTN. c/o tingling in feet x 2 mo. Denies chest pain.
MEDICATIONS: Metformin 1 g BD.
EXAMINATION: Reduced monofilament sensation both feet. No ulcers.
ASSESSMENT: Possible diabetic peripheral neuropathy. Family history of kidney disease (mother).
PLAN: Refer for DPN testing. Review in 3 mo."""),
    ("P1", 3, """HISTORY: 61F with T2DM, HTN. Tingling persists. DPN test done, reduced sural response.
MEDICATIONS: Metformin 1 g BD. Pregabalin 75 mg OD started.
EXAMINATION: Reduced monofilament sensation both feet. No ulcers.
ASSESSMENT: Diabetic peripheral neuropathy confirmed.
PLAN: Foot care advice. Review in 3 mo."""),
    ("P1", 4, """HISTORY: 61F with T2DM, HTN. Tingling better on pregabalin. No SOB.
MEDICATIONS: Metformin 1 g BD. Pregabalin 75 mg OD.
EXAMINATION: No ulcers. Pedal pulses present.
ASSESSMENT: Diabetic peripheral neuropathy, stable.
PLAN: Continue. Review in 6 mo."""),
    ("P2", 1, """HISTORY: 45M, new T2DM. No chest pain. Father had a heart attack at 52.
MEDICATIONS: None.
EXAMINATION: BMI 33. Pedal pulses present.
ASSESSMENT: Type 2 diabetes, new diagnosis. Query fatty liver on scan.
PLAN: Start metformin 500 mg OD. Diet advice."""),
    ("P2", 2, """HISTORY: 45M with T2DM. Mild nausea on metformin. No chest pain.
MEDICATIONS: Metformin 500 mg BD.
EXAMINATION: BMI 32.
ASSESSMENT: Type 2 diabetes. Probable MASLD.
PLAN: Liver fibrosis score. Review in 3 mo."""),
    ("P2", 3, """HISTORY: 45M with T2DM. Mild nausea on metformin. No chest pain.
MEDICATIONS: Metformin 500 mg BD.
EXAMINATION: BMI 32.
ASSESSMENT: Type 2 diabetes. Probable MASLD.
PLAN: Liver fibrosis score. Review in 3 mo."""),
    ("P2", 4, """HISTORY: 45M with T2DM. Nausea resolved. Denies SOB.
MEDICATIONS: Metformin 1 g BD.
EXAMINATION: BMI 31.
ASSESSMENT: Type 2 diabetes. MASLD, low fibrosis risk.
PLAN: Review in 6 mo."""),
    ("P3", 1, """HISTORY: 70F with T2DM x 20 yrs, CKD 3. Left heel ulcer x 2 wk. No fever.
MEDICATIONS: Insulin glargine 20 u. Amlodipine 5 mg OD.
EXAMINATION: 2 cm ulcer left heel, no signs of infection. Pedal pulses absent.
ASSESSMENT: Diabetic foot ulcer. Possible peripheral arterial disease.
PLAN: Urgent podiatry. Arterial Doppler."""),
    ("P3", 2, """HISTORY: 70F with T2DM, CKD 3. Ulcer slowly healing. No fever.
MEDICATIONS: Insulin glargine 20 u. Amlodipine 5 mg OD.
EXAMINATION: 1 cm ulcer left heel, clean. Pedal pulses absent.
ASSESSMENT: Diabetic foot ulcer, improving. Peripheral arterial disease on Doppler.
PLAN: Vascular referral. Weekly dressings."""),
    ("P3", 3, """HISTORY: 70F with T2DM, CKD 3. Ulcer slowly healing. No fever.
MEDICATIONS: Insulin glargine 20 u. Amlodipine 5 mg OD.
EXAMINATION: 1 cm ulcer left heel, clean. Pedal pulses absent.
ASSESSMENT: Diabetic foot ulcer, improving. Peripheral arterial disease on Doppler.
PLAN: Vascular referral. Weekly dressings."""),
    ("P3", 4, """HISTORY: 70F with T2DM, CKD 3. Ulcer healed. Denies rest pain.
MEDICATIONS: Insulin glargine 18 u. Amlodipine 5 mg OD.
EXAMINATION: Heel skin intact. Pedal pulses absent.
ASSESSMENT: Healed ulcer. Peripheral arterial disease.
PLAN: Foot protection service. Review in 3 mo."""),
    ("P4", 1, """HISTORY: 29F, 26 weeks pregnant. Family history of diabetes (mother and sister).
MEDICATIONS: Folic acid.
EXAMINATION: BP 118/76.
ASSESSMENT: Gestational diabetes on OGTT.
PLAN: Diet advice. Home glucose monitoring."""),
    ("P4", 2, """HISTORY: 29F, 28 weeks pregnant. Fasting readings above target. No headache.
MEDICATIONS: Folic acid.
EXAMINATION: BP 120/78. No oedema.
ASSESSMENT: Gestational diabetes, not controlled on diet.
PLAN: Start insulin at night."""),
    ("P4", 3, """HISTORY: 29F, 32 weeks pregnant. Readings at target. No hypos.
MEDICATIONS: Isophane insulin 8 u at night.
EXAMINATION: BP 122/80.
ASSESSMENT: Gestational diabetes, controlled.
PLAN: Continue. Growth scan."""),
    ("P4", 4, """HISTORY: Postnatal, 6 weeks. Insulin stopped after delivery.
MEDICATIONS: None.
EXAMINATION: BP 116/74.
ASSESSMENT: Previous gestational diabetes. Rule out type 2 diabetes.
PLAN: HbA1c at 12 weeks postpartum."""),
    ("P5", 1, """HISTORY: 38M with T1DM since age 12. 3 hypos a week, mostly at night. No seizures.
MEDICATIONS: Basal-bolus insulin.
EXAMINATION: Injection sites: lipohypertrophy abdomen.
ASSESSMENT: Recurrent hypoglycaemia. Lipohypertrophy.
PLAN: Rotate sites. Reduce basal 10%. CGM."""),
    ("P5", 2, """HISTORY: 38M with T1DM. Hypos fewer since basal reduced. No seizures.
MEDICATIONS: Basal-bolus insulin.
EXAMINATION: Lipohypertrophy smaller.
ASSESSMENT: Hypoglycaemia improving.
PLAN: CGM review in 2 wk."""),
    ("P5", 3, """HISTORY: 38M with T1DM. Hypos fewer since basal reduced. No seizures.
MEDICATIONS: Basal-bolus insulin.
EXAMINATION: Lipohypertrophy smaller.
ASSESSMENT: Hypoglycaemia improving.
PLAN: CGM review in 2 wk."""),
    ("P5", 4, """HISTORY: 38M with T1DM. No hypos in last month. Brother has coeliac disease.
MEDICATIONS: Basal-bolus insulin.
EXAMINATION: Injection sites normal.
ASSESSMENT: Type 1 diabetes, stable. ?coeliac screen.
PLAN: Coeliac serology. Review in 3 mo."""),
]
print(f"{len(clinic_notes)} notes for {len({p for p, _, _ in clinic_notes})} synthetic patients")

## Exercise

**Goal:** Build a small clinical-text preprocessing pipeline and test where rules break.

**Data:** Twenty synthetic clinic notes written for the notebook (Created for this course; no real patient data)

### Step 1

Reproduce section splitting, abbreviation expansion and negation detection on all twenty notes.

In [ ]:
# Your code here


### Step 2

Add rules for uncertainty ('probable', 'possible', 'query') and family history ('mother', 'father', 'family history of').

In [ ]:
# Your code here


### Step 3

Write five notes where your rules fail and explain why.

In [ ]:
# Your code here


### Step 4

Measure copy-forward between consecutive notes for each synthetic patient and flag notes that are more than 90% identical.

In [ ]:
# Your code here


### Step 5

Optional: run medspaCy's ConText on the same notes and compare.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/working-with-clinical-notes](https://ramkumarmd.com/medical-ai/clinical-nlp/working-with-clinical-notes). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.